

**THIS IS NOT THE COMPLETE TUTORIAL - see file with (MAIN) in the name. Paste all this code before the first Python block**

First you'll need to select which hardware setup you have. You'll need to select both a `SCOPETYPE` and a `PLATFORM`. `SCOPETYPE` can either be `'OPENADC'` for the CWLite/CW1200 or `'CWNANO'` for the CWNano. `PLATFORM` is the target device, with `'CWLITEARM'`/`'CW308_STM32F3'` being the best supported option, followed by `'CWLITEXMEGA'`/`'CW308_XMEGA'`, then by `'CWNANO'`. As of CW 5.4, you can select the SimpleSerial version
used. For example:

```python
SCOPETYPE = 'OPENADC'
PLATFORM = 'CWLITEARM'
SS_VER = 'SS_VER_2_1'
```

In [1]:
SCOPETYPE = 'OPENADC'
PLATFORM = 'CWLITEARM'
SS_VER = 'SS_VER_2_1'

This code will connect the scope and do some basic setup. We're now just going to use a special setup script to do this. This script contains the commands we ran seperately before.

In [3]:
%run "/home/boyang/chipwhisperer/jupyter/Setup_Scripts/Setup_Generic.ipynb"

INFO: Found ChipWhisperer😍
scope.gain.mode                          changed from low                       to high                     
scope.gain.gain                          changed from 0                         to 30                       
scope.gain.db                            changed from 5.5                       to 24.8359375               
scope.adc.basic_mode                     changed from low                       to rising_edge              
scope.adc.samples                        changed from 24400                     to 5000                     
scope.adc.trig_count                     changed from 16868591                  to 38894856                 
scope.clock.adc_src                      changed from clkgen_x1                 to clkgen_x4                
scope.clock.adc_freq                     changed from 0                         to 29538459                 
scope.clock.adc_rate                     changed from 0.0                       to 29538459.0        

The following code will build the firmware for the target.

In [4]:
%%bash -s "$PLATFORM" "$SS_VER"
cd /home/boyang/chipwhisperer/firmware/mcu/basic-passwdcheck
make PLATFORM=$1 CRYPTO_TARGET=NONE SS_VER=$2 -j

SS_VER set to SS_VER_2_1
SS_VER set to SS_VER_2_1
arm-none-eabi-gcc (15:10.3-2021.07-4) 10.3.1 20210621 (release)
Copyright (C) 2020 Free Software Foundation, Inc.
This is free software; see the source for copying conditions.  There is NO
warranty; not even for MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.

mkdir -p objdir-CWLITEARM 
.
Welcome to another exciting ChipWhisperer target build!!
.
.
.
Compiling:
.
Compiling:
Compiling:
-en     basic-passwdcheck.c ...
Compiling:
-en     .././simpleserial/simpleserial.c ...
-en     .././hal/hal.c ...
.
-en     .././hal//stm32f3/stm32f3_hal.c ...
Compiling:
.
-en     .././hal//stm32f3/stm32f3_hal_lowlevel.c ...
.
Compiling:
Assembling: .././hal//stm32f3/stm32f3_startup.S
-en     .././hal//stm32f3/stm32f3_sysmem.c ...
arm-none-eabi-gcc -c -mcpu=cortex-m4 -I. -x assembler-with-cpp -mthumb -mfloat-abi=soft -fmessage-length=0 -ffunction-sections -DF_CPU=7372800 -Wa,-gstabs,-adhlns=objdir-CWLITEARM/stm32f3_startup.lst -I.././simpleserial/ -

Finally, all that's left is to program the device, which can be done with the following line:

In [5]:
cw.program_target(scope, prog, "/home/boyang/chipwhisperer/firmware/mcu/basic-passwdcheck/basic-passwdcheck-{}.hex".format(PLATFORM))

Detected known STMF32: STM32F302xB(C)/303xB(C)
Extended erase (0x44), this can take ten seconds or more
Attempting to program 4811 bytes at 0x8000000
STM32F Programming flash...
STM32F Reading flash...
Verified flash OK, 4811 bytes


To make interacting with the hardware easier, let's define a function to attempt a password and return a power trace:

In [6]:
def cap_pass_trace(pass_guess):
    reset_target(scope)
    num_char = target.in_waiting()
    while num_char > 0:
        target.read(num_char, 10)
        time.sleep(0.01)
        num_char = target.in_waiting()

    scope.arm()
    target.write(pass_guess)
    ret = scope.capture()
    if ret:
        print('Timeout happened during acquisition')

    trace = scope.get_last_trace()
    return trace

We also don't need all of the default 5000 samples in the trace. 3000 is a good starting point for most targets:

In [8]:
scope.adc.samples = 150

In [9]:
trace_test = cap_pass_trace("h\n")

#Basic sanity check
assert(len(trace_test) == 150)
print("✔️ OK to continue!")

cw.plot(trace_test).opts(color='blue')

✔️ OK to continue!


:Curve   [x]   (y)

In [10]:
trace_test2 = cap_pass_trace("a\n")

#Basic sanity check
assert(len(trace_test) == 150)
print("✔️ OK to continue!")

cw.plot(trace_test2).opts(color='green')

✔️ OK to continue!


:Curve   [x]   (y)

In [11]:
trace_test3 = cap_pass_trace("b\n")

#Basic sanity check
assert(len(trace_test) == 150)
print("✔️ OK to continue!")

cw.plot(trace_test3).opts(color='red')

✔️ OK to continue!


:Curve   [x]   (y)

In [16]:
trace_test4 = cap_pass_trace("c\n")

#Basic sanity check
assert(len(trace_test) == 150)
print("✔️ OK to continue!")

cw.plot(trace_test4).opts(color='orange')

✔️ OK to continue!


:Curve   [x]   (y)

In [17]:
cw.plot(trace_test).opts(color='blue') * cw.plot(trace_test2).opts(color='green') * cw.plot(trace_test3).opts(color='red') * cw.plot(trace_test4).opts(color='orange')

:Overlay
   .Curve.I   :Curve   [x]   (y)
   .Curve.II  :Curve   [x]   (y)
   .Curve.III :Curve   [x]   (y)
   .Curve.IV  :Curve   [x]   (y)